## 1 - Imports

In [1]:
from data_processing.dataloader import get_BAA_DFS
from data_processing.dataset import BAA_Dataset
from data_processing.preprocessing import get_transforms
from model.model import BoneAgeEfficientNet
from model.gradcam import GradCAM

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from os import path, getcwd, makedirs

import cv2 as cv
from PIL import Image
import time

In [2]:
"""from google.colab import drive
drive.mount('/content/drive')"""

"from google.colab import drive\ndrive.mount('/content/drive')"

In [3]:
# cd "drive/MyDrive/tests-sara"

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # i have limited GPU for now and haven't tested with it yet

In [6]:
device = torch.device('cpu') 

## 2 - Preparing Data

In [7]:
batch_size = 64
num_workers = 3

target_image_size = (1024,1024)

apply_segmentation = True
path_to_data = path.join(getcwd(), "data")
img_folder = "masks"

input_dir = path.join(path_to_data, img_folder)

output_dir = path.join(path_to_data, "ROIS")
makedirs(output_dir, exist_ok=True)

In [8]:
tf = get_transforms(target_size = target_image_size, include_resize = True, color_channels_nb = 3)
train_df, val_df = get_BAA_DFS(datapath = path_to_data, image_folder = img_folder, apply_segmentation = apply_segmentation)

val_dataset = BAA_Dataset(val_df, tf, apply_segmentation)
train_dataset = BAA_Dataset(train_df, tf, apply_segmentation)



finished checking

Full df size len: 4226
train/val sizes: 3380/846


## 3 - Preparing Model

In [ ]:
model = BoneAgeEfficientNet(model_name = "tf_efficientnet_b4.ns_jft_in1k", hidden_dim = 1024).to(device)
model_type = 'efficientnet'

checkpoint_path = path.join(getcwd(), "model", "checkpoints")
specific_checkpoint = "efficientnet_b4_1024x1024_aug_geo_best.pth"

p = path.join(checkpoint_path, specific_checkpoint)
if path.exists(p):
    checkpoint = torch.load(p, weights_only=False) #, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'])
    print("checkpoint loaded successfully")
else:
    print("checkpoint doesn't exist")

## 4 - Functions

In [10]:
def get_cam_to_fullres_scales(fullres_img, img, cam):

    img = img.squeeze(0) # .cpu()
    img = img.permute(1, 2, 0)
    full_np = np.asarray(fullres_img)
    
    full_width = full_np.shape[1]
    full_height = full_np.shape[0]

    
    cam_to_resized_scale = img.shape[0] / cam.squeeze().detach().numpy().shape[0]

    resized_to_full_scale_x = full_width / img.shape[0]
    resized_to_full_scale_y = full_height / img.shape[1]

    scale_x = cam_to_resized_scale * resized_to_full_scale_x
    scale_y = cam_to_resized_scale * resized_to_full_scale_y

    return scale_x, scale_y, full_width, full_height

In [11]:
def compute_region_scores(cam, window_size=5):
    # cam: (1,1,H,W)

    kernel = torch.ones( (1, 1, window_size, window_size), device=cam.device )

    scores = F.conv2d( cam, kernel, padding=window_size // 2 )
    return scores.squeeze().detach().cpu().numpy()

In [12]:
def compute_binary_mask(scores_np):
    threshold = np.quantile(scores_np, thresh_quantile)

    binary_mask = (scores_np > threshold).astype(np.uint8)
    binary_mask = binary_mask * 255
    return binary_mask

In [13]:
def determine_crop_regions(contours, fullres_img, scale_x, scale_y, full_width, full_height):

    crops = []

    for contour in contours:
    
        x, y, w, h = cv.boundingRect(contour)
    
        x1 = int(x * scale_x) * crop_ratio
        y1 = int(y * scale_y) * crop_ratio
        
        x2 = int((x + w) * scale_x) / crop_ratio
        y2 = int((y + h) * scale_y) / crop_ratio
    
        x1 = max(0, x1)
        y1 = max(0, y1)
        
        x2 = min(full_width, x2)
        y2 = min(full_height, y2)
        
        crop = fullres_img.crop((x1, y1, x2, y2))
        crops.append(crop)
        
    return crops

In [14]:
def determine_roi_centers(contours, scale_x, scale_y):
    centers = []

    for contour in contours:
        x, y, w, h = cv.boundingRect(contour)

        cx = x + (w / 2)
        cy = y + (h / 2)

        csx = int(cx * scale_x)
        csy = int(cy * scale_y)

        centers.append( (csx, csy) )

    return centers

In [15]:
def crop_from_center(center, fullres_img):
    x,y = center
    half = crop_size / 2
    crop = fullres_img.crop(( x-half, y-half, x+half, y+half) )
    return crop

In [16]:
def process_image(img, fullres_img, gender, age):

    u_img = img.unsqueeze(0).to(device)
    u_gender = gender.unsqueeze(0).to(device)

    pred = model(u_img, u_gender)
    cam = grad_cam.generate(u_img, u_gender).cpu()

    scale_x, scale_y, full_width, full_height = get_cam_to_fullres_scales(fullres_img, img, cam)


    scores_np = compute_region_scores(cam, window_size = window_size) # shape 1,1,32,32

    binary_mask = compute_binary_mask(scores_np)

    contours, _ = cv.findContours(
        binary_mask,
        cv.RETR_EXTERNAL,
        cv.CHAIN_APPROX_SIMPLE
    )

    centers = determine_roi_centers(contours, scale_x, scale_y)
    # crops = determine_crop_regions(contours, fullres_img, scale_x, scale_y, full_width, full_height)

    return centers


In [17]:
def store_results(centers, fullres_img, idx, filename = None, save_imgs = False):
    entries = []
        
    for center, i in zip(centers, range(len(centers))):

        j = { "center_fullres_coords": center }
    
        if save_imgs:
    
                p = path.join(output_dir, f"{filename}_{i}.png")
    
                j = { "center_fullres_coords": center, "roi_path": p }
                
                crop = crop_from_center(center, fullres_img)   
                crop.save(p)
            
        entries.append(j)

    row_idx = ndf.index[ndf["id"] == idx][0]
    ndf.at[row_idx, "roi_info"] = [entries]
    
    ndf.to_csv(path_to_csv, index = False, mode='w')

## 5 - Main Loop and Params

In [18]:
window_size = 3
thresh_quantile = 0.95 

save_images = True

crop_size = 512
#crop_ratio = 0.80 # 1.0 is bbox size, the more you reduce the bigger it is from bbox

In [19]:
target_layer = model.backbone.conv_head
grad_cam = GradCAM(model, target_layer)

In [20]:
def main(df, dataset, csv_name):

    alr_done = 0
    dsnt_exist = 0
    n = dataset.__len__()
    
    in_filename_base = ".png"
    if apply_segmentation:
        in_filename_base = "_segmented.png"
    
    t0 = time.time()
    
    for i in range(n):
    
        img, gender, age, idx = dataset.__getitem__(i)
        
        row = df.loc[df["id"] == idx]
        
        fullres_img = Image.open(row["img_path"].item()).convert("RGB")
    
        in_filename = f"{idx}{in_filename_base}"
        out_filename = f"{idx}_cropped"
    
    
        if not path.exists(path.join(input_dir, in_filename)):
            dsnt_exist += 1
        else:
            if not path.exists(path.join(output_dir, f"{out_filename}_0.png")):
                centers = process_image(img, fullres_img, gender, age)
                store_results(centers, fullres_img, idx, out_filename, save_imgs = save_images)
            else:
                alr_done += 1
    
        print(f"Processed: {i+1} / {n}. already exists: {alr_done}, problematic df entries: {dsnt_exist}. Running time: {(time.time() - t0):.1f}s.", end='\r')
    
    ndf.to_csv(path_to_csv, index = False, mode='w')
    print(f"Done. {path_to_data}")

In [21]:
df = train_df
dataset = train_dataset
csv_name = "train_dataset_with_rois.csv"
path_to_csv = path.join(path_to_data, csv_name)

if not path.exists(path_to_csv):

    ndf = df.copy(deep=True)
    ndf["roi_info"] = None
    ndf.to_csv(path_to_csv, index = False, mode='w')

else:

    ndf = pd.read_csv(path_to_csv, delimiter=",")

        
output_dir = path.join(path_to_data, "ROIS", "train")
makedirs(output_dir, exist_ok=True)

main(df, dataset, csv_name)

Processed: 25 / 3380. already exists: 0, problematic df entries: 0. Running time: 320.3s.

KeyboardInterrupt: 